## SENG 533 — Custom CNN Architecture
### Setup, Instrumentation and Controlled Experiment

In [ ]:
import os
import random
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Logging Setup
import logging

logger = logging.getLogger(__name__)

# Remove any existing handlers before adding new ones
if logger.hasHandlers():
    logger.handlers.clear()

logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

### 1 · Instrumentation & Reproducibility Helpers

In [ ]:
import time
import csv

LOG_FILE = "performance_logs.csv"

# ── Reproducibility ───
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    """Ensures DataLoader workers are seeded for full reproducibility."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# ── GPU Device Setup ───
def get_gpu_type():
    """Returns the available GPU device or CPU if none are available."""
    if torch.cuda.is_available():
        device = torch.cuda.get_device_name(0)
        logger.info(f"GPU found: {torch.cuda.get_device_name(0)}. Using GPU.")

    else:
        device = torch.device("cpu")
        logger.info("No GPU found. Using CPU.")
    return device

def get_gpu_memory():
    """Returns (current_allocated_MB, peak_allocated_MB) for the current GPU."""
    if torch.cuda.is_available():
        current_allocated = torch.cuda.memory_allocated(0) / (1024 ** 2)  # Convert to MB
        peak_allocated = torch.cuda.max_memory_allocated(0) / (1024 ** 2)  # Convert to MB
        return round(current_allocated, 2), round(peak_allocated, 2)
    else:
        return 0, 0

def reset_gpu_memory():
    """Resets GPU memory tracking to get accurate measurements for the next operation."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def cuda_sync():
    """Ensures all CUDA operations are completed before measuring time or memory."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()

# Clear existing log file for a fresh start
if os.path.exists(LOG_FILE):
    os.remove(LOG_FILE)
    print(f"Cleared existing {LOG_FILE} for a fresh experiment run.")

# ── Performance Logging ───
def init_log():
    if not os.path.exists(LOG_FILE):
        with open(LOG_FILE, mode='w', newline='') as file:
            csv.writer(file).writerow([
                'model', 'batch_size', 'num_epochs', 'gpu_type',
                'run_number', 'total_training_time_sec', 'avg_time_per_epoch_sec',
                'inference_latency_single_img_sec', 'inference_latency_batch_sec',
                'throughput_img_per_sec',
                'gpu_memory_allocated_MB', 'gpu_memory_peak_MB',
                'validation_accuracy_pct', 'f1_score'

            ])
    print(f"Performance log initialized at {LOG_FILE}\n Logging will be saved to this file.")

def log_performance(model_name, batch_size, num_epochs, gpu_type, run_num,
                    total_time, avg_epoch, lat_single, lat_batch, throughput,
                    mem_used, mem_peak, val_acc, f1):
    with open(LOG_FILE, mode='a', newline='') as file:
        csv.writer(file).writerow([
            model_name, batch_size, num_epochs, gpu_type,
            run_num, round(total_time, 2), round(avg_epoch, 2),
            round(lat_single, 4), round(lat_batch, 4), round(throughput, 2),
            mem_used, mem_peak,
            round(val_acc * 100, 2), round(f1 * 100, 2)
        ])

    logger.info(f"Logged performance for {model_name} (Run {run_num}) to {LOG_FILE}")
    logger.info(f" Logged: bs={batch_size} | epochs={num_epochs} | gpu={gpu_type} ")

seed_everything(42)
init_log()
logger.info(f'GPU Type: {get_gpu_type()}')


### 2 · Shared Preprocessing Pipeline

In [ ]:
import cv2
import imutils
from PIL import Image
import torchvision
from torchvision import transforms

## Normalization - Standard for both Models
NET_MEAN = [0.485, 0.456, 0.406]
NET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224 # Fixed Resolution for both models

def crop_img(img):
    """
    Crops out the brain region from an MRI scan using contour detection.
    Applied as the first preprocessing step for both models.
    This function assumes the brain is the largest contour in the image and crops to a square around it.
    Args:
        img (numpy array): Input MRI image as a NumPy array.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) # PIL images are in RGB format, not BGR
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
    thresh = cv2.erode(thresh, None, iterations=2)
    thresh = cv2.dilate(thresh, None, iterations=2)

    cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = imutils.grab_contours(cnts)

    c = max(cnts, key=cv2.contourArea)
    extLeft = tuple(c[c[:, :, 0].argmin()][0])
    extRight = tuple(c[c[:, :, 0].argmax()][0])
    extTop = tuple(c[c[:, :, 1].argmin()][0])
    extBot = tuple(c[c[:, :, 1].argmax()][0])

    return img[extTop[1]:extBot[1], extLeft[0]:extRight[0]].copy()

class CropTransform:
    """Custom transform to crop the brain region from MRI scans."""
    def __call__(self, img):
        img_np = np.array(img)
        try:
            return Image.fromarray(crop_img(img_np))
        except Exception as e:
            logger.warning(f"Crop failed: {e}. Returning original image.")
            return img # If cropping fails, return the original image to avoid breaking the pipeline.

# Shared Transformations for both models
# Train: Crop -> Resize -> RandomHorizontalFlip -> ToTensor -> Normalize
# Val/Test: Crop -> Resize -> ToTensor -> Normalize : No data augmentation during validation/testing to ensure consistent evaluation.

TRAIN_TRANSFORMS = transforms.Compose([
    CropTransform(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=NET_MEAN, std=NET_STD)
])

VAL_TEST_TRANSFORMS = transforms.Compose([
    CropTransform(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NET_MEAN, std=NET_STD)
])

logger.info("Shared Preprocessing and Augmentation Pipeline defined for both models.")
logger.info("Data transformations defined for training and validation/testing.")

logger.info(f"Resolution: {IMG_SIZE}x{IMG_SIZE} ")
logger.info(f"Normalization: mean={NET_MEAN}")
logger.info(f"Normalization: std={NET_STD}")
logger.info("Data augmentation: RandomHorizontalFlip applied only during training.")
logger.info("CropTransform applied to all datasets to focus on brain region.")



### 3 · Dataset & DataLoader Builder

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler, random_split

# ── Download dataset ──────
DATASET_DIR = kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset')
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
logger.info(f"Dataset downloaded from Kaggle: {DATASET_DIR}")
logger.info(f"Classes: {CLASSES}")

# ── Build base datasets ────
# Load the full Training folder (no transforms yet — just to get targets/indices)
_full_train_dataset = torchvision.datasets.ImageFolder(
    root=os.path.join(DATASET_DIR, 'Training'),
    transform=None
)

# ── Analyze class distribution ────
# 80/20 split of the full training dataset to create a validation set while maintaining class distribution.
_train_size = int(0.8 * len(_full_train_dataset))
_val_size = len(_full_train_dataset) - _train_size
_generator = torch.Generator().manual_seed(42)  # For reproducibility
_train_idx, _val_idx = random_split(
    range(len(_full_train_dataset)), [_train_size, _val_size], generator=_generator
)
logger.info(f"Full training dataset size: {len(_full_train_dataset)}")
logger.info(f"Training set size: {_train_size}")
logger.info(f"Validation set size: {_val_size}")

TRAIN_INDICES = list(_train_idx)
VAL_INDICES = list(_val_idx)

# Weighted  sampler targets - derived from splits to ensure the same class distribution in the weighted sampler as in the training set.
_train_targets = [_full_train_dataset.targets[i] for i in TRAIN_INDICES]
_class_counts = torch.bincount(torch.tensor(_train_targets))
_class_weights = 1.0 / _class_counts.float()
SAMPLE_WEIGHTS = [_class_weights[target] for target in _train_targets]

logger.info(f"Class counts: {_class_counts.tolist()}")
logger.info(f"Class weights: {_class_weights.tolist()}")


def make_loaders(batch_size):
    """
    Build fresh train/val/test DataLoaders for a given batch size.
    Uses shared TRAIN/VAL_TRANSFORMS, TRAIN/VAL_INDICES, and SAMPLE_WEIGHTS.
    Seeded workers ensure reproducibility across runs.
    """
    g = torch.Generator()
    g.manual_seed(42)

    # Train dataset with transforms
    _train_dataset = torch.utils.data.Subset(
        torchvision.datasets.ImageFolder(
            root=os.path.join(DATASET_DIR, 'Training'), transform=TRAIN_TRANSFORMS),
            TRAIN_INDICES
    )

    # Validation dataset with transforms
    _val_dataset = torch.utils.data.Subset(
        torchvision.datasets.ImageFolder(
            root=os.path.join(DATASET_DIR, 'Training'), transform=VAL_TEST_TRANSFORMS),
            VAL_INDICES
    )

    # Test dataset with transforms
    _test_dataset = torchvision.datasets.ImageFolder(
        root=os.path.join(DATASET_DIR, 'Testing'), transform=VAL_TEST_TRANSFORMS
    )

    # Weighted sampler for training set
    _train_sampler = WeightedRandomSampler(
        weights=SAMPLE_WEIGHTS,
        num_samples=len(SAMPLE_WEIGHTS),
        replacement=True,
    )

    # DataLoaders
    train_loader = DataLoader(
        _train_dataset, batch_size=batch_size, sampler=_train_sampler,
        num_workers=2, pin_memory=True,
        worker_init_fn=seed_worker, generator=g
    )

    val_loader = DataLoader(
        _val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True,
        worker_init_fn=seed_worker, generator=g
    )

    test_loader = DataLoader(
        _test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True,
        worker_init_fn=seed_worker, generator=g
    )

    return train_loader, val_loader, test_loader

logger.info("DataLoaders will be created on demand for each model and batch size using the make_loaders function.")

### 4 · Model Architecture

In [ ]:
# Custom CNN Model Definition

class CustomCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(CustomCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(256 * 14 * 14, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = self.pool(self.relu(self.bn4(self.conv4(x))))
        x = x.view(-1, 256 * 14 * 14)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        return self.fc3(x)


_device = torch.device('cuda') if torch.cuda.is_available() else torch.device("cpu")
_model = CustomCNN(num_classes=len(CLASSES)).to(_device)
_x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(_device)
assert _model(_x).shape == (1, len(CLASSES)), "Model output shape mismatch. Expected (1, num_classes)."
logger.info("Custom CNN model defined and output shape verified with a dummy input.")
del _model, _x

### 5 · Full Experiment Loop

In [ ]:
# 18 configurations × 3 runs = 54 total runs.
# All metrics logged to performance_logs.csv.

import torch.optim as optim
from sklearn.metrics import f1_score as sk_f1_score

BATCH_SIZES = [16, 32, 64]
EPOCH_COUNTS = [5, 10, 20] # 3 different epoch counts to evaluate the impact of training duration on performance and resource usage.
NUM_RUNS = 3
MODEL_NAME = "CustomCNN"
LR = 0.001

device = torch.device('cuda') if torch.cuda.is_available() else torch.device("cpu")
gpu_type = get_gpu_type()

logger.info(f"Beginning training and evaluation for {MODEL_NAME} on Device: {device} - GPU: {gpu_type}")
logger.info(f"Starting training and evaluation for {MODEL_NAME} across configurations.")

best_val_acc = 0.0  # Track the best validation accuracy across all runs for summary logging at the end.
best_model_path = "best_custom_cnn.pth"  # Path to save the best model based on validation accuracy.

for batch_size in BATCH_SIZES:
    for num_epochs in EPOCH_COUNTS:
        for run in range(1, NUM_RUNS + 1):
            logger.info(f"{MODEL_NAME} - Run {run}/{NUM_RUNS} | Batch Size: {batch_size} | Epochs: {num_epochs}")

            reset_gpu_memory()

            train_loader, val_loader, test_loader = make_loaders(batch_size)
            model = CustomCNN(num_classes=len(CLASSES)).to(device)
            criterion = torch.nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=LR)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.1, patience=3
            )

            # ── Training Loop ────
            epoch_times = []
            train_start_time = time.time()
            y_true_last, y_pred_last = [], []  # For final F1 score calculation on validation set
            last_epoch_val_loss = float('inf')

            for epoch in range(1, num_epochs + 1):
                model.train()
                epoch_start_time = time.time()

                for images, labels in train_loader:
                    images, labels = images.to(device), labels.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(images), labels)
                    loss.backward()
                    optimizer.step()

                cuda_sync() # Ensure all GPU operations are complete before timing
                epoch_times.append(time.time() - epoch_start_time)

                # ── Validation Loop ────
                model.eval()
                val_loss, val_correct, val_total = 0.0, 0, 0
                y_true, y_pred = [], []
                with torch.no_grad():
                    for images, labels in val_loader:
                        images, labels = images.to(device), labels.to(device)
                        outputs = model(images)
                        val_loss += criterion(outputs, labels).item()
                        _, predicted = torch.max(outputs.data, 1)
                        val_total += labels.size(0)
                        val_correct += (predicted == labels).sum().item()
                        y_true.extend(labels.cpu().numpy())
                        y_pred.extend(predicted.cpu().numpy())

                val_loss /= len(val_loader)
                scheduler.step(val_loss) # Adjust learning rate based on validation loss
                last_val_acc = 100.0 * val_correct / val_total
                y_true_last, y_pred_last = y_true, y_pred # Store for final F1 score calculation
                logger.info(f"Epoch {epoch}/{num_epochs} - Time: {epoch_times[-1]:.2f}s - Val Loss: {val_loss:.4f} - Val Acc: {last_val_acc:.2f}%")

                if last_val_acc > best_val_acc:
                    best_val_acc = last_val_acc
                    torch.save(model.state_dict(), best_model_path)
                    logger.info(f"New best model saved with validation accuracy: {best_val_acc:.2f}%")

            cuda_sync() # Ensure all GPU operations are complete before timing
            total_training_time = time.time() - train_start_time
            avg_time_per_epoch = np.mean(epoch_times)
            mem_current, mem_peak = get_gpu_memory() # Get GPU memory usage after training is complete to capture peak usage during training. Peak mem = max memory allocated at any point during training.
            final_f1 = sk_f1_score(y_true_last, y_pred_last, average='weighted')

            # ── Inference Timing on Validation Set ────
            model.eval()
            bench_batch, _ = next(iter(val_loader))
            bench_batch = bench_batch.to(device)
            single_img = bench_batch[:1]

            with torch.no_grad():
                # Warm-up - not measured during evaluation to ensure accurate timing
                cuda_sync()
                _ = model(single_img)
                cuda_sync()


                # Single image Latency - measured on a single image to capture the overhead of model loading and initial inference.
                # Averaged over 100 runs to get a stable estimate.
                _times = []
                for _ in range(100):
                    cuda_sync()
                    start_time = time.time()
                    _ = model(single_img)
                    cuda_sync()
                    _times.append(time.time() - start_time)
                lat_single = np.mean(_times)

                # Batch Latency - measured on a full batch to capture the model's performance under typical inference conditions.
                cuda_sync()
                start_time = time.time()
                _ = model(bench_batch)
                cuda_sync()
                lat_batch = time.time() - start_time
                throughput = len(bench_batch) / lat_batch if lat_batch > 0 else 0

            log_performance(
                model_name=MODEL_NAME,
                batch_size=batch_size,
                num_epochs=num_epochs,
                gpu_type=gpu_type,
                run_num=run,
                total_time=total_training_time,
                avg_epoch=avg_time_per_epoch,
                lat_single=lat_single,
                lat_batch=lat_batch,
                throughput=throughput,
                mem_used=mem_current,
                mem_peak=mem_peak,
                val_acc=last_val_acc,
                f1=final_f1 * 100  # Convert to percentage for logging
            )

logger.info(f"Completed all runs for {MODEL_NAME}. Performance metrics logged to {LOG_FILE}.")
logger.info(f'Best validation accuracy across all runs: {best_val_acc:.2f}%')

### 6 · Final Evaluation on Test Set

In [ ]:
## After all runs are complete, we can analyze the logged performance metrics in performance_logs.csv to compare the impact of batch size and epoch count on training time, inference latency, GPU memory usage, and model performance (accuracy and F1 score).

device = torch.device('cuda') if torch.cuda.is_available() else torch.device("cpu")
_, _, test_loader = make_loaders(batch_size=32) # Use a standard batch size for testing

model = CustomCNN(num_classes=len(CLASSES)).to(device)
model.load_state_dict(torch.load("best_custom_cnn.pth", map_location=device)) # Load the best model checkpoint saved during training

model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

logger.info(classification_report(y_true, y_pred, target_names=CLASSES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Custom CNN - Confusion Matrix on Test Set')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()